# Test shortcut weights

## Librairies

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Go to project root (adjust depth if needed)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data")
sys.path.append(ROOT)

print("Added to path:", ROOT)

import torch
from torchvision import datasets, transforms

from src.models.networks import FashionMLP_Large
from src.quantization.quantize import quantize_model
from src.shortcuts.shortcut_weights import *

Added to path: /Users/jeremiecabessa/Desktop/ROOT/Articles/Conference_Papers/2025_With_Jiri/code/ErrorVolumePolytopes


In [2]:
torch.manual_seed(42)

## Large MLP (FashionMNIST)

In [3]:
# ============== #
# Model and Data #
# ============== #

# Device
device = torch.device("gpu" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

# Model
model_name = "fashion_mlp_best.pth"
model_path = os.path.join(ROOT, "checkpoints", model_name)

model = FashionMLP_Large()
model.load_state_dict(torch.load(model_path))#['model_state'])
model.to(device).eval()

train_dataset = torch.load(
    DATA_PATH + "/fashionMNIST_correct_mlp.pt",
    weights_only=False
)

# _, test_dataset = load_mnist_datasets()
x_0, c = train_dataset[123]
x_0 = x_0.flatten().unsqueeze(0) # shape (1, input_dim)
x_0 = x_0.to(device) # important
print("Sample x_0 shape:", x_0.shape)
print("min and max of x_0:", x_0.min().item(), x_0.max().item())
print("Models and dataset have been loaded.")

Sample x_0 shape: torch.Size([1, 784])
min and max of x_0: -1.0 1.0
Models and dataset have been loaded.


In [4]:
def test_shortcut_weights_notebook(model, x_0):

    # =========================== #
    # Weights and Biases on model #
    # =========================== #

    weights, biases = get_weights_and_biases(model)

    print("*** Weights and Biases ***\n")
    for i, (W, b) in enumerate(zip(weights, biases)):
        print(f"Layer {i}: W shape = {W.shape}, b shape = {b.shape}")


    # =========================== #
    # Shortcut weights and Biases #
    # =========================== #

    print("\n*** Shortcut Weights and Biases ***\n")

    print("(a) Testing shortcut weights on small MLP\n")

    W_shortcut_l, B_shortcut_l, unsaturations_l = compute_shortcut_weights(model, x_0)

    for i, (W, B) in enumerate(zip(W_shortcut_l, B_shortcut_l)):
        print(f"Layer {i}: W_shortcut shape = {W.shape}", 
                f"\tB_shortcut shape = {B.shape}")


    W_shortcut, unsaturation_mask = pack_shortcut_weights(W_shortcut_l, B_shortcut_l, unsaturations_l)
    print(f"\nPacked shortcut weights shape: {W_shortcut.shape}")
    print(f"Packed unsaturation mask shape: {unsaturation_mask.shape}")
    print(unsaturation_mask[-10:], "\n")

    test_shortcut_weights(model, x_0)

test_shortcut_weights_notebook(model, x_0)

*** Weights and Biases ***

Layer 0: W shape = torch.Size([1024, 784]), b shape = torch.Size([1024])
Layer 1: W shape = torch.Size([512, 1024]), b shape = torch.Size([512])
Layer 2: W shape = torch.Size([256, 512]), b shape = torch.Size([256])
Layer 3: W shape = torch.Size([128, 256]), b shape = torch.Size([128])
Layer 4: W shape = torch.Size([10, 128]), b shape = torch.Size([10])

*** Shortcut Weights and Biases ***

(a) Testing shortcut weights on small MLP

Layer 0: W_shortcut shape = torch.Size([1024, 784]) 	B_shortcut shape = torch.Size([1024])
Layer 1: W_shortcut shape = torch.Size([512, 784]) 	B_shortcut shape = torch.Size([512])
Layer 2: W_shortcut shape = torch.Size([256, 784]) 	B_shortcut shape = torch.Size([256])
Layer 3: W_shortcut shape = torch.Size([128, 784]) 	B_shortcut shape = torch.Size([128])
Layer 4: W_shortcut shape = torch.Size([10, 784]) 	B_shortcut shape = torch.Size([10])

Packed shortcut weights shape: torch.Size([1930, 785])
Packed unsaturation mask shape: to

In [5]:
qmodel = quantize_model(model, bits=4)
test_shortcut_weights_notebook(qmodel, x_0)

*** Weights and Biases ***

Layer 0: W shape = torch.Size([1024, 784]), b shape = torch.Size([1024])
Layer 1: W shape = torch.Size([512, 1024]), b shape = torch.Size([512])
Layer 2: W shape = torch.Size([256, 512]), b shape = torch.Size([256])
Layer 3: W shape = torch.Size([128, 256]), b shape = torch.Size([128])
Layer 4: W shape = torch.Size([10, 128]), b shape = torch.Size([10])

*** Shortcut Weights and Biases ***

(a) Testing shortcut weights on small MLP

Layer 0: W_shortcut shape = torch.Size([1024, 784]) 	B_shortcut shape = torch.Size([1024])
Layer 1: W_shortcut shape = torch.Size([512, 784]) 	B_shortcut shape = torch.Size([512])
Layer 2: W_shortcut shape = torch.Size([256, 784]) 	B_shortcut shape = torch.Size([256])
Layer 3: W_shortcut shape = torch.Size([128, 784]) 	B_shortcut shape = torch.Size([128])
Layer 4: W_shortcut shape = torch.Size([10, 784]) 	B_shortcut shape = torch.Size([10])

Packed shortcut weights shape: torch.Size([1930, 785])
Packed unsaturation mask shape: to